In [11]:
import os
import glob
import numpy as np
import geopandas as gpd
import pandas as pd
import rasterio
import laspy
from shapely.geometry import box

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

####################
# 1. Read training data & match TIF/ LAS
####################

# fix new training data
# 1. Read new_training
new_gdf = gpd.read_file("/Users/iosefa/Downloads/TrainingSquare.gpkg")
new_gdf = new_gdf.to_crs(epsg=32605)

# 2. Assign values to the new 'class' column based on 'code'
def code_to_class(code):
    if code == "METPOL":
        return 1
    elif code == "GREROB":
        return 2
    elif code == "ACAKOA":
        return 3
    elif code == "DIOSAN":
        return 4
    else:
        return 6

new_gdf['class'] = new_gdf['code'].apply(code_to_class)
new_gdf = new_gdf[["class", "geometry"]]

# 3. Read training
training_gdf = gpd.read_file("/Users/iosefa/trial_small/training.gpkg")
training_gdf = training_gdf[["class", "geometry"]]

# 4. Concatenate (join) the two dataframes
all_data = pd.concat([training_gdf, new_gdf], ignore_index=True)

# 5. Write result to new file
all_gdf = gpd.GeoDataFrame(all_data, geometry='geometry', crs=training_gdf.crs)
all_gdf.to_file("/Users/iosefa/trial_small/new_training.gpkg", driver="GPKG")



TRAINING_GPKG = "/Users/iosefa/trial_small/new_training.gpkg"
INDIVIDUAL_CROWNS_DIR = "/Users/iosefa/trial_small/individual_crowns"
INDIVIDUAL_CROWNS_POINTS_DIR = "/Users/iosefa/trial_small/individual_crowns_points"

gdf_training = gpd.read_file(TRAINING_GPKG)

tif_files = sorted(glob.glob(os.path.join(INDIVIDUAL_CROWNS_DIR, "*.tif")))
las_files = sorted(glob.glob(os.path.join(INDIVIDUAL_CROWNS_POINTS_DIR, "*.las")))

tif_dict = {}
for tif_path in tif_files:
    base = os.path.splitext(os.path.basename(tif_path))[0]
    tif_dict[base] = tif_path

las_dict = {}
for las_path in las_files:
    base = os.path.splitext(os.path.basename(las_path))[0]
    las_dict[base] = las_path

common_keys = set(tif_dict.keys()).intersection(las_dict.keys())
training_records = []

for key in common_keys:
    tif_path = tif_dict[key]
    las_path = las_dict[key]

    with rasterio.open(tif_path) as src:
        left, bottom, right, top = src.bounds
    tif_bbox = box(left, bottom, right, top)

    candidate_points = gdf_training[gdf_training.geometry.within(tif_bbox)]
    if len(candidate_points) == 0:
        continue
    label = candidate_points["class"].mode()[0]
    training_records.append((tif_path, las_path, label))


In [24]:
# ####################
# # 2. PyTorch Dataset
# ####################
# from PIL import Image

# class CrownDataset(Dataset):
#     def __init__(self, records, img_transform=None, max_points=2048):
#         self.records = records
#         self.img_transform = img_transform
#         self.max_points = max_points

#     def __len__(self):
#         return len(self.records)

#     def __getitem__(self, idx):
#         tif_path, las_path, label = self.records[idx]

#         # Load image
#         with rasterio.open(tif_path) as src:
#             img = src.read()  # shape: (bands, height, width)
#         img = torch.from_numpy(img).float()  # shape: (C, H, W)

#         # If the image is multispectral with 9 bands (WorldView-3 with CHM),
#         # select bands 5, 3, and 2 (1-indexed) which correspond to indices 4, 2, and 1.
#         if img.shape[0] == 9:
#             img = img[[4, 2, 1], :, :]

#         # if self.img_transform:
#         #     # If using additional transforms (e.g., normalization), apply here.
#         #     pass
        
#         # Then actually apply transform in __getitem__
#         if self.img_transform:
#             img = np.moveaxis(img.numpy(), 0, -1)  # shape: (H, W, C) for Pillow
#             img = Image.fromarray(img.astype(np.uint8))  # if indeed 0..255
#             img = self.img_transform(img)  

#         # Load LAS (point cloud)
#         with laspy.open(las_path) as f:
#             las = f.read()
#             x = las.x
#             y = las.y
#             z = las.z
#         points = np.vstack((x, y, z)).T  # shape: (N, 3)

#         if points.shape[0] > self.max_points:
#             choice = np.random.choice(points.shape[0], self.max_points, replace=False)
#             points = points[choice, :]

#         points = torch.from_numpy(points).float()  # shape: (N, 3)

#         return {
#             'image': img,
#             'points': points,
#             'label': label
#         }

# # img_transform = transforms.Compose([
# #     transforms.ToTensor(),
# # ])
# img_transform = transforms.Compose([
#             transforms.ToTensor(),  # converts H,W,C [0..255] to float [0..1]
#             transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                                  std=[0.229, 0.224, 0.225])
#         ])
# dataset = CrownDataset(training_records, img_transform=img_transform)

# # DataLoader returns batched data, so each tensor already has a batch dimension.
# dataloader = DataLoader(dataset, batch_size=1, shuffle=False)


# ####################
# # 3. Define PyTorch Feature Extractors (ResNet + PointNet)
# ####################

# class ResNetFeatureExtractor(nn.Module):
#     def __init__(self, model_name='resnet18', pretrained=True):
#         super().__init__()
#         if model_name == 'resnet18':
#             self.model = models.resnet18(pretrained=pretrained)
#             feat_size = 512
#         elif model_name == 'resnet50':
#             self.model = models.resnet50(pretrained=pretrained)
#             feat_size = 2048
#         else:
#             raise ValueError("Unsupported model_name")
#         # Remove final FC layer
#         self.model.fc = nn.Identity()
#         self._out_features = feat_size

#     def forward(self, x):
#         return self.model(x)

#     @property
#     def output_dim(self):
#         return self._out_features


# class PointNetFeatureExtractor(nn.Module):
#     def __init__(self, input_dim=3, hidden_dim=64):
#         super().__init__()
#         self.fc1 = nn.Linear(input_dim, hidden_dim)
#         self.fc2 = nn.Linear(hidden_dim, hidden_dim * 2)
#         self.fc3 = nn.Linear(hidden_dim * 2, hidden_dim * 4)
#         self.fc4 = nn.Linear(hidden_dim * 4, hidden_dim * 2)
#         self.fc5 = nn.Linear(hidden_dim * 2, hidden_dim)
#         self._out_features = hidden_dim

#     def forward(self, x):
#         # x: (B, N, 3)
#         batch_size, num_points, _ = x.shape
#         x = F.relu(self.fc1(x))
#         x = F.relu(self.fc2(x))
#         x = F.relu(self.fc3(x))
#         # Max pooling across points
#         x = torch.max(x, dim=1)[0]  # shape: (B, hidden_dim*4)
#         x = F.relu(self.fc4(x))     # shape: (B, hidden_dim*2)
#         x = F.relu(self.fc5(x))     # shape: (B, hidden_dim)
#         return x

#     @property
#     def output_dim(self):
#         return self._out_features


# ####################
# # 4. Inference-Mode Feature Extraction
# ####################
# device = torch.device("mps")

# image_feature_extractor = ResNetFeatureExtractor('resnet18', pretrained=True).to(device)
# point_feature_extractor = PointNetFeatureExtractor(input_dim=3, hidden_dim=64).to(device)

# # Set to eval mode
# image_feature_extractor.eval()
# point_feature_extractor.eval()

# X_features = []
# y_labels = []

# with torch.no_grad():
#     for batch in dataloader:
#         # DataLoader returns batched tensors:
#         #   img: shape (1, C, H, W)
#         #   pts: shape (1, N, 3)
#         img = batch['image'].to(device)
#         pts = batch['points'].to(device)
#         label = batch['label'].item()

#         # img already has shape (1, 3, H, W) after selecting RGB bands
#         img_feats = image_feature_extractor(img)  # Expected shape: (1, 512)
#         pt_feats = point_feature_extractor(pts)     # Expected shape: (1, 64)

#         # Concatenate the features along dimension 1
#         combined_feat = torch.cat([img_feats, pt_feats], dim=1)  # shape: (1, 576)

#         # Convert to NumPy array and remove the batch dimension
#         combined_feat_np = combined_feat.cpu().numpy().squeeze(axis=0)  # shape: (576,)
#         X_features.append(combined_feat_np)
#         y_labels.append(label)

# # Convert lists to NumPy arrays for scikit-learn
# X_features = np.array(X_features)  # shape: (num_samples, 576)
# y_labels = np.array(y_labels)

# print("Feature matrix shape:", X_features.shape)
# print("Labels shape:", y_labels.shape)

# ####################
# # 5. Train scikit-learn MLP
# ####################
# ####################
# # 5. Train scikit-learn MLP with stratified split and remapped classes
# ####################
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import accuracy_score, classification_report

# # Remap classes: from original labels {1, 2, 4, 6} to {0, 1, 2, 3}
# class_mapping = {1: 0, 2: 1, 4: 2, 6: 3}
# y_mapped = np.array([class_mapping[label] for label in y_labels])

# print("Original label distribution:")
# for label in sorted(class_mapping.keys()):
#     print(f"Class {label}: {np.sum(y_labels == label)}")

# print("\nRemapped label distribution:")
# for new_label in np.unique(y_mapped):
#     print(f"New Class {new_label}: {np.sum(y_mapped == new_label)}")

# # Stratified split so each set gets a fair share of every class
# X_train, X_test, y_train, y_test = train_test_split(
#     X_features, y_mapped, test_size=0.3, random_state=42, stratify=y_mapped
# )

# print(f"\nTrain samples: {len(y_train)}, Test samples: {len(y_test)}")

# # classified_mlp = classify(
# #     ob, tr, acceptable_classes_gdf=regions, test_size=0.2, compute_reports=True,
# #     method='mlp', compute_shap=False, sample_shap=True, hidden_layer_sizes=(200, 150, 100,),
# #     solver='lbfgs', max_iter=10000
# # )

# mlp = MLPClassifier(hidden_layer_sizes=(256, 128, ),
#                     activation='relu',
#                     solver='adam',
#                     max_iter=10000,
#                     random_state=42)

# mlp.fit(X_train, y_train)

# # Evaluate
# y_pred = mlp.predict(X_test)
# acc = accuracy_score(y_test, y_pred)
# print("\nTest accuracy:", acc)
# print("\nClassification Report:")
# print(classification_report(y_test, y_pred))

####################
# 2. PyTorch Dataset
####################
from PIL import Image

def min_max_scale_per_band(band):
    """
    Scale a single band (2D array) to [0..255] using min–max normalization.
    """
    min_val = band.min()
    max_val = band.max()
    if max_val == min_val:
        # Avoid dividing by zero if the entire band is a constant
        return np.zeros_like(band, dtype=np.float32)

    scaled = (band - min_val) / (max_val - min_val)
    scaled *= 255.0
    return scaled

class CrownDataset(Dataset):
    def __init__(self, records, img_transform=None, max_points=2048):
        self.records = records
        self.img_transform = img_transform
        self.max_points = max_points

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        tif_path, las_path, label = self.records[idx]

        # --- 1. Load image from TIF ---
        with rasterio.open(tif_path) as src:
            # shape: (bands, H, W)
            img = src.read()  

        # If the image is multispectral with 9 bands (WorldView-3 + CHM),
        # select bands 5, 3, and 2 (1-indexed) which correspond to indices [4, 2, 1].
        if img.shape[0] == 9:
            img = img[[4, 2, 1], :, :]

        # Convert to float32 for safe manipulation
        img = img.astype(np.float32)

        # --- 2. Min–max scale each band to [0..255] ---
        for b in range(img.shape[0]):
            img[b] = min_max_scale_per_band(img[b])

        # --- 3. Convert (bands, H, W) -> (H, W, bands), then to uint8 ---
        img = np.moveaxis(img, 0, -1)   # shape: (H, W, C)
        img = img.astype(np.uint8)

        # --- 4. Convert to PIL image, apply transforms (if any) ---
        pil_img = Image.fromarray(img)  
        if self.img_transform:
            pil_img = self.img_transform(pil_img)  
            # shape: (3, H, W) after transforms.ToTensor()

        # --- 5. Load LAS (point cloud) ---
        with laspy.open(las_path) as f:
            las = f.read()
            x = las.x
            y = las.y
            z = las.z
        points = np.vstack((x, y, z)).T  # shape: (N, 3)

        if points.shape[0] > self.max_points:
            choice = np.random.choice(points.shape[0], self.max_points, replace=False)
            points = points[choice, :]

        # Convert to Torch tensor
        points = torch.from_numpy(points).float()  # (N, 3)

        return {
            'image': pil_img,   # a Tensor if img_transform includes ToTensor
            'points': points,
            'label': label
        }


####################
# Data Transform
####################
img_transform = transforms.Compose([
    # This expects [0..255] input as a PIL image, yields [0..1]
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

dataset = CrownDataset(training_records, img_transform=img_transform)
dataloader = DataLoader(dataset, batch_size=1, shuffle=False)

####################
# 3. ResNet + PointNet
####################
class ResNetFeatureExtractor(nn.Module):
    def __init__(self, model_name='resnet18', pretrained=True):
        super().__init__()
        if model_name == 'resnet18':
            self.model = models.resnet18(pretrained=pretrained)
            feat_size = 512
        elif model_name == 'resnet50':
            self.model = models.resnet50(pretrained=pretrained)
            feat_size = 2048
        else:
            raise ValueError("Unsupported model_name")
        # Remove final FC layer
        self.model.fc = nn.Identity()
        self._out_features = feat_size

    def forward(self, x):
        return self.model(x)

    @property
    def output_dim(self):
        return self._out_features


class PointNetFeatureExtractor(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=64):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim * 2)
        self.fc3 = nn.Linear(hidden_dim * 2, hidden_dim * 4)
        self.fc4 = nn.Linear(hidden_dim * 4, hidden_dim * 2)
        self.fc5 = nn.Linear(hidden_dim * 2, hidden_dim)
        self._out_features = hidden_dim

    def forward(self, x):
        # x: (B, N, 3)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        # Max pooling across points dimension
        x = torch.max(x, dim=1)[0]  # shape: (B, hidden_dim*4)
        x = F.relu(self.fc4(x))     # shape: (B, hidden_dim*2)
        x = F.relu(self.fc5(x))     # shape: (B, hidden_dim)
        return x

    @property
    def output_dim(self):
        return self._out_features

####################
# 4. Feature Extraction
####################
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

image_feature_extractor = ResNetFeatureExtractor('resnet18', pretrained=True).to(device)
point_feature_extractor = PointNetFeatureExtractor(input_dim=3, hidden_dim=64).to(device)

image_feature_extractor.eval()
point_feature_extractor.eval()

X_features = []
y_labels = []

with torch.no_grad():
    for batch in dataloader:
        img = batch['image'].to(device)  # shape: (1, 3, H, W)
        pts = batch['points'].to(device) # shape: (1, N, 3)
        label = batch['label'].item()

        img_feats = image_feature_extractor(img)  # -> (1, 512) for ResNet18
        pt_feats = point_feature_extractor(pts)   # -> (1, 64) for the small PointNet

        combined_feat = torch.cat([img_feats, pt_feats], dim=1)  # shape: (1, 576)

        # Convert to NumPy array (removing batch dimension)
        combined_feat_np = combined_feat.cpu().numpy().squeeze(0)  # (576,)
        X_features.append(combined_feat_np)
        y_labels.append(label)

X_features = np.array(X_features)
y_labels = np.array(y_labels)

print("Feature matrix shape:", X_features.shape)
print("Labels shape:", y_labels.shape)

####################
# 5. Train scikit-learn MLP
####################
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

class_mapping = {1: 0, 2: 1, 4: 2, 6: 3}
y_mapped = np.array([class_mapping[l] for l in y_labels])

print("Original label distribution:")
for label in sorted(class_mapping.keys()):
    print(f"Class {label}: {np.sum(y_labels == label)}")

print("\nRemapped label distribution:")
for new_label in np.unique(y_mapped):
    print(f"New Class {new_label}: {np.sum(y_mapped == new_label)}")

X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_mapped, test_size=0.3, random_state=42, stratify=y_mapped
)

print(f"\nTrain samples: {len(y_train)}, Test samples: {len(y_test)}")

mlp = MLPClassifier(hidden_layer_sizes=(256, 128),
                    activation='relu',
                    solver='adam',
                    max_iter=10000,
                    random_state=42)

mlp.fit(X_train, y_train)

# Evaluate
y_pred = mlp.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print("\nTest accuracy:", acc)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

/Users/iosefa/repos/PyForestScan/venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/iosefa/repos/PyForestScan/venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Feature matrix shape: (1391, 576)
Labels shape: (1391,)
Original label distribution:
Class 1: 225
Class 2: 556
Class 4: 492
Class 6: 118

Remapped label distribution:
New Class 0: 225
New Class 1: 556
New Class 2: 492
New Class 3: 118

Train samples: 973, Test samples: 418

Test accuracy: 0.39952153110047844

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        68
           1       0.40      1.00      0.57       167
           2       0.00      0.00      0.00       148
           3       0.00      0.00      0.00        35

    accuracy                           0.40       418
   macro avg       0.10      0.25      0.14       418
weighted avg       0.16      0.40      0.23       418



/Users/iosefa/repos/PyForestScan/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/iosefa/repos/PyForestScan/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/iosefa/repos/PyForestScan/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.

In [1]:
# import time

# import geopandas as gpd
import numpy as np

# from rasterio.mask import mask

from obia.handlers.geotif import open_geotiff
import geopandas as gpd
from obia.segmentation.segment_statistics import create_objects

img = open_geotiff('/Users/iosefa/trial_small/mid-elevation.tif')
print("Image shape:", img.img_data.shape)
# Transpose from (height, width, bands) to (bands, height, width)
img.img_data = np.transpose(img.img_data, (2, 0, 1))
print("After transpose:", img.img_data.shape)  # should show (9, 1167, 1758)

segments = gpd.read_file("/Users/iosefa/trial_small/merged_crowns.gpkg")

objects_gdf = create_objects(
    segments, img, ept='/Users/iosefa/ept/ept.json',
    ept_srs='EPSG:32605', textural_bands=[0], voxel_resolution=(500,500,2),
    calculate_structural=True, calculate_radiometric=True
)

objects_gdf.to_file('/Users/iosefa/trial_small/objects_test_ept.gpkg')
objects_gdf

Image shape: (1167, 1758, 9)
After transpose: (9, 1167, 1758)


Processing Segments:   9%|▉         | 228/2602 [00:10<01:57, 20.22it/s]/Users/iosefa/repos/obia/obia/segmentation/segment_statistics.py:175: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  stats_dict[f"{band_prefix}_skewness"] = skew(band_flat)
/Users/iosefa/repos/obia/obia/segmentation/segment_statistics.py:177: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  stats_dict[f"{band_prefix}_kurtosis"] = kurtosis(band_flat)
Processing Segments:  31%|███▏      | 817/2602 [00:38<01:17, 22.91it/s]/Users/iosefa/repos/obia/obia/segmentation/segment_statistics.py:175: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  stats_dict[f"

,segment_id,b0_mean,b0_variance,b0_min,b0_max,b0_skewness,b0_kurtosis,b1_mean,b1_variance,b1_min,...,b0_homogeneity,b0_ASM,b0_energy,b0_correlation,pai,fhd,ch,mean_intensity,variance_intensity,geometry
0,4687.0,688.857685,8732.209348,372.0,943.0,-0.339215,0.421921,622.299810,19055.811442,324.0,...,0.820250,0.663865,0.814569,0.597427,0.512001,1.648066,12.590064,4994.091549,539561.606642,"MULTIPOLYGON (((201041.5 2191741, 201041.5 219..."
1,4775.0,706.882852,6055.193407,420.0,974.0,-0.521770,1.110982,748.101868,10170.172985,424.0,...,0.857265,0.736734,0.857652,0.125625,0.592535,1.474289,9.150000,5476.769793,525351.005470,"MULTIPOLYGON (((201051 2191732, 201050.5 21917..."
2,4855.0,704.801117,9463.129161,372.0,1006.0,-0.096630,0.616428,720.939665,16271.933790,291.0,...,0.943473,0.890525,0.943582,0.171183,0.595190,1.689761,12.971055,4951.260264,510432.383202,"MULTIPOLYGON (((201049.5 2191721, 201049 21917..."
3,4938.0,702.173333,8760.211860,356.0,958.0,-0.233498,0.015201,715.518095,11481.506815,324.0,...,0.820688,0.666716,0.816196,0.553261,0.483422,1.743722,14.658862,4995.005030,677032.461745,"MULTIPOLYGON (((201061.5 2191715.5, 201061 219..."
4,4653.0,679.781333,10017.525518,277.0,895.0,-0.568244,0.733050,627.194667,12770.402105,341.0,...,0.931632,0.868542,0.931813,0.143442,0.404389,1.499228,11.199302,4520.587644,347702.681974,"MULTIPOLYGON (((201110 2191753, 201110.5 21917..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2597,2224.0,675.556391,3551.314489,499.0,832.0,-0.576619,0.738201,643.849624,7460.924755,424.0,...,0.553182,0.299227,0.541519,0.305576,1.011313,1.460395,11.818593,4474.618462,145719.275967,"MULTIPOLYGON (((201044 2192024.5, 201044 21920..."
2598,2468.0,750.288660,3952.184717,515.0,863.0,-0.848975,0.876509,821.639175,5313.653311,590.0,...,0.249884,0.053175,0.224681,0.457697,0.571631,1.087099,5.430000,6098.244541,842627.328846,"MULTIPOLYGON (((201090 2192003, 201090 2192002..."
2599,2933.0,697.920635,6434.178886,420.0,895.0,-0.301180,0.796108,760.031746,13556.940791,507.0,...,0.908636,0.826791,0.909009,0.139856,0.461977,1.275213,7.013056,5785.950382,698692.978454,"MULTIPOLYGON (((201104.5 2191958, 201105 21919..."
2600,2952.0,752.409449,5972.367785,515.0,927.0,-0.205901,-0.279437,794.110236,9026.680761,574.0,...,0.263832,0.061055,0.246276,0.535661,0.451280,1.284846,7.130270,5065.815668,616552.251736,"MULTIPOLYGON (((201092.5 2191960, 201092.5 219..."


In [2]:
import geopandas as gpd

from obia.utils.utils import label_segments
from obia.classification.classify import classify

objects = gpd.read_file("/Users/iosefa/trial_small/objects_test_ept.gpkg")
ob = objects.dropna().reset_index(drop=True).copy()
ob

/Users/iosefa/repos/PyForestScan/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,segment_id,b0_mean,b0_variance,b0_min,b0_max,b0_skewness,b0_kurtosis,b1_mean,b1_variance,b1_min,...,b0_homogeneity,b0_ASM,b0_energy,b0_correlation,pai,fhd,ch,mean_intensity,variance_intensity,geometry
0,4687.0,688.857685,8732.209348,372.0,943.0,-0.339215,0.421921,622.299810,19055.811442,324.0,...,0.820250,0.663865,0.814569,0.597427,0.512001,1.648066,12.590064,4994.091549,539561.606642,"MULTIPOLYGON (((201041.5 2191741, 201041.5 219..."
1,4775.0,706.882852,6055.193407,420.0,974.0,-0.521770,1.110982,748.101868,10170.172985,424.0,...,0.857265,0.736734,0.857652,0.125625,0.592535,1.474289,9.150000,5476.769793,525351.005470,"MULTIPOLYGON (((201051 2191732, 201050.5 21917..."
2,4855.0,704.801117,9463.129161,372.0,1006.0,-0.096630,0.616428,720.939665,16271.933790,291.0,...,0.943473,0.890525,0.943582,0.171183,0.595190,1.689761,12.971055,4951.260264,510432.383202,"MULTIPOLYGON (((201049.5 2191721, 201049 21917..."
3,4938.0,702.173333,8760.211860,356.0,958.0,-0.233498,0.015201,715.518095,11481.506815,324.0,...,0.820688,0.666716,0.816196,0.553261,0.483422,1.743722,14.658862,4995.005030,677032.461745,"MULTIPOLYGON (((201061.5 2191715.5, 201061 219..."
4,4653.0,679.781333,10017.525518,277.0,895.0,-0.568244,0.733050,627.194667,12770.402105,341.0,...,0.931632,0.868542,0.931813,0.143442,0.404389,1.499228,11.199302,4520.587644,347702.681974,"MULTIPOLYGON (((201110 2191753, 201110.5 21917..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2593,2224.0,675.556391,3551.314489,499.0,832.0,-0.576619,0.738201,643.849624,7460.924755,424.0,...,0.553182,0.299227,0.541519,0.305576,1.011313,1.460395,11.818593,4474.618462,145719.275967,"MULTIPOLYGON (((201044 2192024.5, 201044 21920..."
2594,2468.0,750.288660,3952.184717,515.0,863.0,-0.848975,0.876509,821.639175,5313.653311,590.0,...,0.249884,0.053175,0.224681,0.457697,0.571631,1.087099,5.430000,6098.244541,842627.328846,"MULTIPOLYGON (((201090 2192003, 201090 2192002..."
2595,2933.0,697.920635,6434.178886,420.0,895.0,-0.301180,0.796108,760.031746,13556.940791,507.0,...,0.908636,0.826791,0.909009,0.139856,0.461977,1.275213,7.013056,5785.950382,698692.978454,"MULTIPOLYGON (((201104.5 2191958, 201105 21919..."
2596,2952.0,752.409449,5972.367785,515.0,927.0,-0.205901,-0.279437,794.110236,9026.680761,574.0,...,0.263832,0.061055,0.246276,0.535661,0.451280,1.284846,7.130270,5065.815668,616552.251736,"MULTIPOLYGON (((201092.5 2191960, 201092.5 219..."


In [3]:
from pyforestscan.handlers import read_lidar
from pyforestscan.calculate import assign_voxels

pointclouds = read_lidar('/Users/iosefa/trial_small/ept/ept.json', srs='EPSG:32605')
voxel_resolution=(500,500,2)

voxels, extent = assign_voxels(pointclouds[0], voxel_resolution)

ValueError: Point cloud data missing 'X', 'Y', or 'HeightAboveGround' fields.

In [28]:
if not pointclouds:
    print("oh no")

In [3]:


# training_points = gpd.read_file("/Users/iosefa/training_points.gpkg")
training_points = gpd.read_file("/Users/iosefa/trial_small/new_training.gpkg")
training_points["feature_class"] = training_points["class"]
training_points

,class,geometry,feature_class
0,2,POINT (200812.191 2192017.438),2
1,2,POINT (201012.5 2192020.738),2
2,2,POINT (200780.351 2191991.899),2
3,2,POINT (200872.823 2191982.955),2
4,2,POINT (200747.23 2191958.498),2
...,...,...,...
2934,2,POINT (201051.947 2191705.103),2
2935,2,POINT (201055.288 2191701.475),2
2936,2,POINT (201048.108 2191710.783),2
2937,2,POINT (201068.613 2191788.232),2


In [5]:
training, mixed = label_segments(ob, training_points)

regions = gpd.read_file("/Users/iosefa/Dropbox/Academic/Research/Projects/Hawaii/Change HI/Unsorted/Puuwaawaa/combined_classes.gpkg")

regions['acceptable_classes'] = regions['acceptable_classes'].apply(
    lambda x: [int(cls) for cls in x.split(',')]
)


In [9]:

columns_tr = [
    'segment_id',
    'b2_mean',
    'b6_mean',
    'b7_mean',
    'b1_skewness',
    'b0_variance',
    'b3_max',
    'b5_max',
    'b0_min',
    'b0_kurtosis',
    'b0_energy',
    'b0_contrast',
    'b0_dissimilarity',
    'pai',
    'fhd',
    'ch',
    'mean_intensity',
    'variance_intensity',
    'feature_class',
    'geometry'
]
tr = training[columns_tr].copy()

columns_ob = [
    'segment_id',
    'b2_mean',
    'b6_mean',
    'b7_mean',
    'b1_skewness',
    'b0_variance',
    'b3_max',
    'b5_max',
    'b0_min',
    'b0_kurtosis',
    'b0_energy',
    'b0_contrast',
    'b0_dissimilarity',
    'pai',
    'fhd',
    'ch',
    'mean_intensity',
    'variance_intensity',
    'geometry'
]

ob = ob[columns_ob]

print("training", tr.shape)

# mlp = MLPClassifier(hidden_layer_sizes=(256, 128),
#                     activation='relu',
#                     solver='adam',
#                     max_iter=500,
#                     random_state=42)

classified_mlp = classify(
    ob, tr, acceptable_classes_gdf=regions, test_size=0.3, compute_reports=True,
    method='mlp', compute_shap=False, sample_shap=True, 
    # hidden_layer_sizes=(200, 150, 100,),
    hidden_layer_sizes=(256, 128,),
    # solver='lbfgs', 
    solver='adam', 
    max_iter=10000
)
# classified_mlp.classified.to_file(
#     "/Users/iosefa/trial_small/classified_mlp_sefa.gpkg"
# )

print(classified_mlp.confusion_matrix)
print(classified_mlp.report)

training (949, 20)
[[ 33   1   3   0]
 [  4  95   5   4]
 [  1   6 101   6]
 [  3   6   6  11]]
              precision    recall  f1-score   support

         1.0       0.80      0.89      0.85        37
         2.0       0.88      0.88      0.88       108
         4.0       0.88      0.89      0.88       114
         6.0       0.52      0.42      0.47        26

    accuracy                           0.84       285
   macro avg       0.77      0.77      0.77       285
weighted avg       0.84      0.84      0.84       285



/Users/iosefa/repos/PyForestScan/venv/lib/python3.12/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
